In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
import itertools as itools
import time
import numpy as np
from collections import defaultdict
from scipy.optimize import linprog
import chaospy as cp
from typing import List, Callable, Union
from numpy.polynomial.legendre import leggauss
import pandas as pd
from pyomo.environ import *

In [2]:
# m = ConcreteModel()
# 
# # y = {'0':1, '1':1, '2':1, '3':1}
# # y = {'0':0.5, '1':0.5, '2':0.5, '3':0.5}
# y = {'0':0, '1':0, '2':0, '3':0}
# 
# suppliers = ['A', 'B', 'C']
# markets = ['D', 'E', 'F']
# sites = ['0', '1', '2', '3']
# 
# supply_theta = {'A': 45000, 'B': 100000, 'C': 75000}
# 
# conv = 1/2.4 # basis is feed
# 
# demand = {'D': 0, 'E': 60000, 'F': 25000} # demand of product
# 
# no_trips = 365 # number of trips made by each truck per year, basically one round trip per day per truck
# 
# site_cap = {'0': 50000, '1': 50000, '2': 50000, '3': 75000} # capacity of each site, basis is feed
# 
# flex_goal = 0.17 # goal for cfri
# 
# availability_factor = 0.959260199
# 
# m.suppliers = Set(initialize=suppliers)
# m.markets = Set(initialize=markets)
# m.sites = Set(initialize=sites)
# 
# # control variables
# m.product = Var(m.sites, domain = NonNegativeReals, doc = 'amount of product made at each site')
# m.feed = Var(m.sites, domain = NonNegativeReals, doc = 'amount of feed used at each site')
# m.feed_ship = Var(m.suppliers, m.sites, domain = NonNegativeReals, doc = 'amount of feed shipped from supplier to site')
# m.product_ship = Var(m.sites, m.markets, domain = NonNegativeReals, doc = 'amount of product shipped from site to market')
# 
# # design variables
# m.trans_cap = Var(m.sites, domain = NonNegativeReals, doc = 'transport capacity around each site')
# m.capex = Var(domain = NonNegativeReals)
# 
# # CFRI
# # m.cfri = Var(domain = NonNegativeReals)
# # 
# # constraints
# def balance_feed(model, site):
#     return model.feed[site] == sum(model.feed_ship[supplier, site] for supplier in model.suppliers)
# m.balance_feed_con = Constraint(m.sites, rule=balance_feed)
# 
# def conversion(model, site):
#     return model.product[site] == model.feed[site]*conv
# m.conversion_con = Constraint(m.sites, rule=conversion)
# 
# def balance_product(model, site):
#     return model.product[site] == sum(model.product_ship[site, market] for market in model.markets)
# m.balance_product_con = Constraint(m.sites, rule=balance_product)
# 
# def transport_cost(model):
#     return model.capex == sum(model.trans_cap[site] for site in model.sites)
# m.transport_cost_con = Constraint(rule=transport_cost)
# 
# def limit_feed(model, supplier):
#     return sum(model.feed_ship[supplier, site] for site in model.sites) - supply_theta[supplier] <= 0
# m.limit_feed_con = Constraint(m.suppliers, rule=limit_feed)
# 
# def limit_production(model, site):
#     return model.feed[site] - site_cap[site]*availability_factor <=0
# m.limit_production_con = Constraint(m.sites, rule=limit_production)
# 
# def limit_demand(model, market):
#     return sum(model.product_ship[site, market] for site in model.sites) >= demand[market]
# m.limit_demand_con = Constraint(m.markets, rule=limit_demand)
# 
# def limit_transport(model, site):
#     return sum(model.feed_ship[supplier, site] for supplier in model.suppliers) + sum(model.product_ship[site, market] for market in markets) - model.trans_cap[site] * no_trips * y[site] <= 0
# m.limit_transport_con = Constraint(m.sites, rule=limit_transport)
# 
# # def cfri_rule(model):
# #     d0_ratio = model.trans_cap[sites[0]]/576
# #     d1_ratio = model.trans_cap[sites[1]]/576
# #     d2_ratio = model.trans_cap[sites[2]]/576
# #     d3_ratio = model.trans_cap[sites[3]]/576
# # 
# #     return coefs[0] + coefs[1]*d0_ratio + coefs[2]*d1_ratio + coefs[3]*d2_ratio + coefs[4]*d3_ratio + coefs[5]*d0_ratio**2 + coefs[6]*d0_ratio*d1_ratio + coefs[7]*d0_ratio*d2_ratio + coefs[8]*d0_ratio*d3_ratio + coefs[9]*d1_ratio**2 + coefs[10]*d1_ratio*d2_ratio + coefs[11]*d1_ratio*d3_ratio + coefs[12]*d2_ratio**2 + coefs[13]*d2_ratio*d3_ratio + coefs[14]*d3_ratio**2 + intercept == model.cfri 
# # m.cfri_con = Constraint(rule=cfri_rule)
# 
# # def flex_rule(model):
# #     return model.cfri >= flex_goal
# # m.flex_con = Constraint(rule=flex_rule)
# 
# # objective
# m.obj = Objective(expr = (m.capex), sense=minimize)
# 
# # solve
# results = SolverFactory('gurobi', solver_io = 'python').solve(m, tee = True)

In [3]:
suppliers = ['A', 'B', 'C']
markets = ['D', 'E', 'F']
sites = ['0', '1', '2', '3']

supply_theta_nominal = {'A': 45000, 'B': 100000, 'C': 75000}
supply_theta_stddev = {'A': np.sqrt(7500), 'B': np.sqrt(16667), 'C': np.sqrt(12500)}

conv = 1/2.4 # basis is feed

demand = {'D': 0, 'E': 60000, 'F': 25000} # demand of product

no_trips = 365 # number of trips made by each truck per year, basically one round trip per day per truck

site_cap = {'0': 50000, '1': 50000, '2': 50000, '3': 75000} # capacity of each site, basis is feed

flex_goal = 0.17 # goal for cfri

availability_factor = 0.959260199

In [4]:
t_bounds = {'A':(supply_theta_nominal['A'] - 4*supply_theta_stddev['A'], supply_theta_nominal['A'] + 4*supply_theta_stddev['A']),
            'B':(supply_theta_nominal['B'] - 4*supply_theta_stddev['B'], supply_theta_nominal['B'] + 4*supply_theta_stddev['B']),
            'C':(supply_theta_nominal['C'] - 4*supply_theta_stddev['C'], supply_theta_nominal['C'] + 4*supply_theta_stddev['C'])}

d_bounds = {'0':(0, 1000), '1':(0, 1000), '2':(0, 1000), '3':(0, 1000)}
nt = len(t_bounds)
nd = len(d_bounds)

In [24]:
# y = {'0':1, '1':1, '2':1, '3':1}
y = {'0':0.5, '1':0.5, '2':0.5, '3':0.5}
# y = {'0':0, '1':0, '2':0, '3':0}

In [25]:
m = MPModeler()

In [26]:
u = m.add_var(name='u')
product = {si: m.add_var(name=f'product[{si}]') for si in sites}
feed = {si: m.add_var(f'feed[{si}]') for si in sites}
feed_ship = {(su, si): m.add_var(name=f'feed_ship[{su},{si}]') for su, si in itools.product(suppliers, sites)}
product_ship = {(si, ma): m.add_var(name=f'product_ship[{si},{m}]') for si, ma in itools.product(sites, markets)}
capex = m.add_var(name='capex')
trans_cap = {si: m.add_var(name=f'trans_cap[{si}]') for si in sites}

supply_theta = {su: m.add_param(name=f'supply_theta[{su}]') for su in suppliers}

In [27]:
# def balance_feed(model, site):
#     return model.feed[site] == sum(model.feed_ship[supplier, site] for supplier in model.suppliers)
# m.balance_feed_con = Constraint(m.sites, rule=balance_feed)

m.add_constrs(sum(feed_ship[su, si] for su in suppliers) == feed[si] for si in sites)

In [28]:
# def transport_cost(model):
#     return model.capex == sum(model.trans_cap[site] for site in model.sites)
# m.transport_cost_con = Constraint(rule=transport_cost)

m.add_constr(capex == sum(trans_cap[si] for si in sites))

In [29]:
# def conversion(model, site):
#     return model.product[site] == model.feed[site]*conv
# m.conversion_con = Constraint(m.sites, rule=conversion)

m.add_constrs(product[si] == feed[si]*conv for si in sites)

In [30]:
# def balance_product(model, site):
#     return model.product[site] == sum(model.product_ship[site, market] for market in model.markets)
# m.balance_product_con = Constraint(m.sites, rule=balance_product)

m.add_constrs(sum(product_ship[si, ma] for ma in markets) == product[si] for si in sites)

In [31]:
# def limit_feed(model, supplier):
#     return sum(model.feed_ship[supplier, site] for site in model.sites) - supply_theta[supplier] <= 0
# m.limit_feed_con = Constraint(m.suppliers, rule=limit_feed)

m.add_constrs(sum(feed_ship[su, si] for si in sites) <= supply_theta[su] + u for su in suppliers)

In [32]:
# def limit_production(model, site):
#     return model.feed[site] - site_cap[site] <=0
# m.limit_production_con = Constraint(m.sites, rule=limit_production)

m.add_constrs(feed[si] <= site_cap[si] * availability_factor + u for si in sites)

In [33]:
# def limit_demand(model, market):
#     return sum(model.product_ship[site, market] for site in model.sites) >= demand[market]
# m.limit_demand_con = Constraint(m.markets, rule=limit_demand)

m.add_constrs(demand[ma] <= sum(product_ship[si, ma] for si in sites) + u for ma in markets)

In [34]:
# def limit_transport(model, site):
#     return sum(model.feed_ship[supplier, site] for supplier in model.suppliers) + sum(model.product_ship[site, market] for market in markets) - model.trans_cap[site] * no_trips <= 0
# m.limit_transport_con = Constraint(m.sites, rule=limit_transport)

m.add_constrs(feed[si] + product[si] <= trans_cap[si]*no_trips*y[si] + u for si in sites)

In [35]:
# theta bounds
m.add_constrs(supply_theta[su] >= t_bounds[su][0] for su in suppliers)
m.add_constrs(supply_theta[su] <= t_bounds[su][1] for su in suppliers)

In [36]:
# design bounds
m.add_constrs(trans_cap[si] >= d_bounds[si][0] for si in sites)
m.add_constrs(trans_cap[si] <= d_bounds[si][1] for si in sites)

In [37]:
m.set_objective(u)

In [38]:
prob = m.formulate_problem()
prob.process_constraints()

In [39]:
# np.hstack((prob.A_t, prob.b_t))

In [40]:
solution_flexibility = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.geometric_parallel_exp)

Using a found active set [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 31, 32, 33, 34]
Spawned threads across 24
 Number of Facets to look at this time 6
 Number of Regions added in this pass 0!


In [41]:
len(solution_flexibility.critical_regions)

1

In [42]:
solution_flexibility

Solution(program=<ppopt.mplp_program.MPLP_Program object at 0x00000245EB3913F0>, critical_regions=[Critical region with active set [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 31, 32, 33, 34]
The Omega Constraint indices are [0, 1, 2, 3, 4, 5]
The Lagrange multipliers Constraint indices are []
The Regular Constraint indices are [[], []]
  x(θ) = Aθ + b 
 λ(θ) = Cθ + d 
  Eθ <= f
 A = [[ 0.00000000e+00 -7.01949199e-16 -7.80779596e-16]
 [ 0.00000000e+00  6.31320153e-17  8.85332072e-17]
 [ 0.00000000e+00  1.60420731e-16  3.85167610e-17]
 [ 0.00000000e+00  1.43423981e-16  3.10464752e-16]
 [ 0.00000000e+00  1.43113990e-16  1.85511931e-16]
 [ 0.00000000e+00 -3.31814220e-16 -3.82740407e-16]
 [ 0.00000000e+00 -1.01153074e-17 -2.19677822e-16]
 [ 0.00000000e+00 -1.68751636e-16 -1.51690824e-16]
 [ 0.00000000e+00 -1.82030406e-16 -5.16579320e-17]
 [ 0.00000000e+00 -2.50000000e-01 -2.50000000e-01]
 [ 0.00000000e+00 -2.50000000e-01 -2.50000000e-01]
 [ 0.00000000e+00 